
# 06 — Visualize Failures

Objectif :

Analyser visuellement les trajectoires de ton agent pour comprendre :

- où il échoue ;
- où il touche l'île ;
- pourquoi certains épisodes sont lents ;
- si le choix gauche/droite est robuste ;
- quelles zones de vent posent problème.

Ce notebook est conçu pour être robuste :

- il détecte automatiquement la racine du dépôt ;
- il charge un agent Python par fichier ;
- il utilise directement l'environnement du dépôt ;
- il sauvegarde les trajectoires et les figures dans `results/failure_analysis/`.


In [ ]:

from pathlib import Path
import sys
import json
import time
import importlib.util
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# 1. Détection de la racine du dépôt


In [ ]:

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src" / "evaluate_submission.py").exists() and (p / "src" / "test_agent_validity.py").exists():
            return p
    raise FileNotFoundError(
        "Impossible de trouver la racine du dépôt. "
        "Lance ce notebook depuis le dossier du repo ou depuis notebooks/."
    )

REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
AGENTS_DIR = SRC_DIR / "agents"
RESULTS_DIR = REPO_ROOT / "results" / "failure_analysis"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("REPO_ROOT  =", REPO_ROOT)
print("SRC_DIR    =", SRC_DIR)
print("AGENTS_DIR =", AGENTS_DIR)
print("RESULTS_DIR=", RESULTS_DIR)



# 2. Imports de l'environnement

On essaie d'importer l'environnement et les scénarios de vent depuis le dépôt.


In [ ]:

try:
    from env_sailing import SailingEnv
    from wind_scenarios import get_wind_scenario
    print("Imported SailingEnv and get_wind_scenario successfully.")
except Exception as e:
    print("Import failed.")
    traceback.print_exc()
    raise



# 3. Fonctions utilitaires robustes


In [ ]:

def load_agent_from_file(agent_path):
    agent_path = Path(agent_path).resolve()
    module_name = f"loaded_agent_{agent_path.stem}_{abs(hash(str(agent_path))) % 10_000_000}"

    spec = importlib.util.spec_from_file_location(module_name, str(agent_path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    if not hasattr(module, "MyAgent"):
        raise AttributeError(f"{agent_path} ne contient pas de classe MyAgent.")

    return module.MyAgent()

def extract_maps_from_observation(obs, grid=128):
    obs = np.asarray(obs)

    expected_wind = grid * grid * 2
    expected_world = grid * grid

    if len(obs) >= 6 + expected_wind + expected_world:
        wind = obs[6:6 + expected_wind].reshape(grid, grid, 2)
        world = obs[6 + expected_wind:6 + expected_wind + expected_world].reshape(grid, grid)
        return wind, world

    return None, None

def get_world_from_env_or_obs(env, obs):
    # Try common attributes first
    for attr in ["world", "world_map", "map", "grid"]:
        if hasattr(env, attr):
            value = getattr(env, attr)
            if isinstance(value, np.ndarray):
                return value

    _, world = extract_maps_from_observation(obs)
    return world

def is_collision_position(pos, world):
    if world is None:
        return False

    x = int(np.clip(round(float(pos[0])), 0, world.shape[1] - 1))
    y = int(np.clip(round(float(pos[1])), 0, world.shape[0] - 1))

    return bool(world[y, x] > 0.5)

def discounted_score(reward_sum, steps, gamma=0.995):
    return float(reward_sum) * (gamma ** int(steps))

def parse_state(obs):
    obs = np.asarray(obs)
    return {
        "x": float(obs[0]),
        "y": float(obs[1]),
        "vx": float(obs[2]),
        "vy": float(obs[3]),
        "wx": float(obs[4]),
        "wy": float(obs[5]),
    }



# 4. Choisir l'agent à analyser

Par défaut, on analyse `src/agents/my_agent.py`.


In [ ]:

candidate_agents = [
    AGENTS_DIR / "my_agent.py",
    AGENTS_DIR / "agent_trained_example.py",
    AGENTS_DIR / "agent_super_naive.py",
]

available_agents = [p for p in candidate_agents if p.exists()]

print("Available agents:")
for i, p in enumerate(available_agents):
    print(i, p.relative_to(REPO_ROOT))

if not available_agents:
    raise FileNotFoundError("Aucun agent trouvé dans src/agents/.")

AGENT_PATH = available_agents[0]
print("\nSelected agent:", AGENT_PATH.relative_to(REPO_ROOT))



# 5. Fonction d'exécution d'un épisode

Cette fonction enregistre :

- position ;
- vitesse ;
- vent local ;
- actions ;
- rewards ;
- collisions ;
- termination/truncation.


In [ ]:

def make_env(scenario_name="training_1", render_mode=None):
    scenario = get_wind_scenario(scenario_name)
    try:
        return SailingEnv(wind_field=scenario, render_mode=render_mode)
    except TypeError:
        try:
            return SailingEnv(scenario, render_mode=render_mode)
        except TypeError:
            return SailingEnv(wind_field=scenario)

def run_episode(agent_path, scenario_name="training_1", seed=0, max_steps=500):
    agent = load_agent_from_file(agent_path)

    if hasattr(agent, "seed"):
        try:
            agent.seed(seed)
        except Exception:
            pass

    if hasattr(agent, "reset"):
        try:
            agent.reset()
        except Exception:
            pass

    env = make_env(scenario_name=scenario_name, render_mode=None)

    reset_out = env.reset(seed=seed)
    if isinstance(reset_out, tuple):
        obs, info = reset_out
    else:
        obs, info = reset_out, {}

    world = get_world_from_env_or_obs(env, obs)

    trajectory = []
    total_reward = 0.0
    reached_goal = False
    collided = False
    error = None

    for step in range(max_steps):
        st = parse_state(obs)
        pos = np.array([st["x"], st["y"]], dtype=float)
        vel = np.array([st["vx"], st["vy"]], dtype=float)
        wind = np.array([st["wx"], st["wy"]], dtype=float)

        coll_now = is_collision_position(pos, world)
        collided = collided or coll_now

        try:
            action = int(agent.act(obs))
        except Exception as e:
            error = repr(e)
            break

        if action < 0 or action > 8:
            error = f"Invalid action {action}"
            break

        try:
            step_out = env.step(action)
        except Exception as e:
            error = repr(e)
            break

        if len(step_out) == 5:
            next_obs, reward, terminated, truncated, info = step_out
        elif len(step_out) == 4:
            next_obs, reward, done, info = step_out
            terminated, truncated = bool(done), False
        else:
            error = f"Unexpected step output length: {len(step_out)}"
            break

        total_reward += float(reward)

        trajectory.append({
            "step": step,
            "x": st["x"],
            "y": st["y"],
            "vx": st["vx"],
            "vy": st["vy"],
            "wx": st["wx"],
            "wy": st["wy"],
            "action": action,
            "reward": float(reward),
            "collision": bool(coll_now),
            "terminated": bool(terminated),
            "truncated": bool(truncated),
        })

        if reward > 0:
            reached_goal = True

        obs = next_obs

        if terminated or truncated:
            break

    df = pd.DataFrame(trajectory)

    steps = len(df)
    score = discounted_score(total_reward, steps)

    summary = {
        "scenario": scenario_name,
        "seed": seed,
        "steps": steps,
        "reward_sum": total_reward,
        "discounted_score": score,
        "success": bool(reached_goal),
        "collided": bool(collided),
        "error": error,
    }

    return summary, df, world



# 6. Tester un épisode unique


In [ ]:

summary, traj, world = run_episode(
    AGENT_PATH,
    scenario_name="training_1",
    seed=0,
    max_steps=500,
)

summary


In [ ]:

display(traj.head())
display(traj.tail())



# 7. Visualiser une trajectoire

La fonction ci-dessous trace :

- la carte ;
- la trajectoire ;
- le départ ;
- le goal ;
- les points de collision si détectés.


In [ ]:

ACTION_NAMES = {
    0: "N",
    1: "NE",
    2: "E",
    3: "SE",
    4: "S",
    5: "SW",
    6: "W",
    7: "NW",
    8: "Stay",
}

def plot_trajectory(traj, world=None, title=None, save_path=None, show_actions=False):
    plt.figure(figsize=(8, 8))

    if world is not None:
        plt.imshow(world, origin="lower", alpha=0.35)
    else:
        plt.xlim(0, 127)
        plt.ylim(0, 127)

    if len(traj) > 0:
        xs = traj["x"].to_numpy()
        ys = traj["y"].to_numpy()

        plt.plot(xs, ys, linewidth=2, marker="o", markersize=2, label="trajectory")
        plt.scatter(xs[0], ys[0], s=100, marker="o", label="start")
        plt.scatter(xs[-1], ys[-1], s=120, marker="x", label="end")

        collisions = traj[traj["collision"] == True]
        if len(collisions) > 0:
            plt.scatter(collisions["x"], collisions["y"], s=200, marker="X", label="collision")

        if show_actions:
            for _, row in traj.iloc[::max(1, len(traj)//20)].iterrows():
                plt.text(row["x"], row["y"], ACTION_NAMES.get(int(row["action"]), str(row["action"])), fontsize=8)

    plt.scatter([64], [127], s=250, marker="*", label="goal")
    plt.title(title or "Trajectory")
    plt.xlim(0, 127)
    plt.ylim(0, 127)
    plt.legend()
    plt.grid(alpha=0.2)

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()

plot_trajectory(
    traj,
    world=world,
    title=f"{summary['scenario']} seed={summary['seed']} | success={summary['success']} | steps={summary['steps']}",
    save_path=RESULTS_DIR / "single_trajectory.png",
)



# 8. Visualiser les actions, vitesses et vent local


In [ ]:

def plot_diagnostics(traj, title=None, save_path=None):
    if len(traj) == 0:
        print("Empty trajectory.")
        return

    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    axes[0].plot(traj["step"], traj["x"], label="x")
    axes[0].plot(traj["step"], traj["y"], label="y")
    axes[0].set_ylabel("position")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(traj["step"], traj["vx"], label="vx")
    axes[1].plot(traj["step"], traj["vy"], label="vy")
    axes[1].set_ylabel("velocity")
    axes[1].legend()
    axes[1].grid(True)

    axes[2].plot(traj["step"], traj["wx"], label="wx")
    axes[2].plot(traj["step"], traj["wy"], label="wy")
    axes[2].set_ylabel("local wind")
    axes[2].legend()
    axes[2].grid(True)

    axes[3].step(traj["step"], traj["action"], where="post")
    axes[3].set_ylabel("action")
    axes[3].set_xlabel("step")
    axes[3].set_yticks(list(ACTION_NAMES.keys()))
    axes[3].set_yticklabels([ACTION_NAMES[i] for i in ACTION_NAMES])
    axes[3].grid(True)

    fig.suptitle(title or "Episode diagnostics")

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()

plot_diagnostics(
    traj,
    title=f"Diagnostics | {summary['scenario']} seed={summary['seed']}",
    save_path=RESULTS_DIR / "single_diagnostics.png",
)



# 9. Lancer plusieurs épisodes et identifier les échecs

On évalue l'agent sur plusieurs seeds et scénarios.

On classe ensuite les épisodes en :

- success ;
- failure ;
- collision ;
- slow success.


In [ ]:

SCENARIOS = ["training_1", "training_2", "training_3"]
SEEDS = list(range(50))

episode_rows = []
trajectories = {}

for scenario_name in SCENARIOS:
    for seed in SEEDS:
        try:
            summary, df, world = run_episode(
                AGENT_PATH,
                scenario_name=scenario_name,
                seed=seed,
                max_steps=500,
            )
        except Exception as e:
            summary = {
                "scenario": scenario_name,
                "seed": seed,
                "steps": np.nan,
                "reward_sum": 0.0,
                "discounted_score": 0.0,
                "success": False,
                "collided": False,
                "error": repr(e),
            }
            df = pd.DataFrame()

        key = (scenario_name, seed)
        trajectories[key] = df

        episode_rows.append(summary)

episodes = pd.DataFrame(episode_rows)

display(episodes.head())
display(
    episodes.groupby("scenario")
    .agg(
        success_rate=("success", "mean"),
        mean_score=("discounted_score", "mean"),
        mean_steps=("steps", "mean"),
        collisions=("collided", "sum"),
        errors=("error", lambda x: sum(v is not None for v in x)),
    )
    .reset_index()
)



# 10. Sauvegarder les diagnostics bruts


In [ ]:

episodes_path = RESULTS_DIR / "episode_summaries.csv"
episodes.to_csv(episodes_path, index=False)

# Save trajectories as one CSV
all_traj_rows = []
for (scenario_name, seed), df in trajectories.items():
    if len(df) == 0:
        continue
    tmp = df.copy()
    tmp["scenario"] = scenario_name
    tmp["seed"] = seed
    all_traj_rows.append(tmp)

if all_traj_rows:
    all_traj = pd.concat(all_traj_rows, ignore_index=True)
else:
    all_traj = pd.DataFrame()

traj_path = RESULTS_DIR / "trajectories.csv"
all_traj.to_csv(traj_path, index=False)

print("Saved:", episodes_path)
print("Saved:", traj_path)



# 11. Sélectionner les épisodes problématiques

Définitions :

- failure : pas de goal ;
- collision : l'agent touche l'île ;
- slow success : succès mais avec beaucoup de steps.


In [ ]:

failures = episodes[episodes["success"] == False].copy()
collisions = episodes[episodes["collided"] == True].copy()

successful = episodes[episodes["success"] == True].copy()
slow_threshold = successful["steps"].quantile(0.90) if len(successful) > 0 else np.inf
slow_success = successful[successful["steps"] >= slow_threshold].copy()

print("Number of failures:", len(failures))
print("Number of collisions:", len(collisions))
print("Slow success threshold:", slow_threshold)
print("Number of slow successes:", len(slow_success))

display(failures.head(10))
display(collisions.head(10))
display(slow_success.sort_values("steps", ascending=False).head(10))



# 12. Visualiser automatiquement les pires trajectoires

On sauvegarde les figures dans `results/failure_analysis/`.


In [ ]:

def get_world_for_scenario(scenario_name):
    env = make_env(scenario_name)
    reset_out = env.reset(seed=0)
    obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out
    return get_world_from_env_or_obs(env, obs)

def plot_episode_from_row(row, label):
    scenario_name = row["scenario"]
    seed = int(row["seed"])
    df = trajectories.get((scenario_name, seed), pd.DataFrame())
    world = get_world_for_scenario(scenario_name)

    if len(df) == 0:
        print("Empty trajectory:", scenario_name, seed)
        return

    fname = f"{label}_{scenario_name}_seed_{seed}.png"
    save_path = RESULTS_DIR / fname

    title = (
        f"{label} | {scenario_name} seed={seed} | "
        f"success={row['success']} steps={row['steps']} score={row['discounted_score']:.2f}"
    )

    plot_trajectory(
        df,
        world=world,
        title=title,
        save_path=save_path,
        show_actions=False,
    )

# Plot up to 5 failures
for _, row in failures.head(5).iterrows():
    plot_episode_from_row(row, "failure")

# Plot up to 5 slow successes
for _, row in slow_success.sort_values("steps", ascending=False).head(5).iterrows():
    plot_episode_from_row(row, "slow_success")

# Plot up to 5 collisions
for _, row in collisions.head(5).iterrows():
    plot_episode_from_row(row, "collision")



# 13. Heatmap des positions visitées

Cela permet de voir :

- si l'agent passe trop près de l'île ;
- s'il utilise toujours le même côté ;
- s'il se bloque dans une zone.


In [ ]:

def visitation_heatmap(traj_df, scenario_name=None, only_failures=False):
    if len(traj_df) == 0:
        print("Empty trajectory dataframe.")
        return

    df = traj_df.copy()

    if scenario_name is not None:
        df = df[df["scenario"] == scenario_name]

    if only_failures:
        fail_keys = set((r["scenario"], int(r["seed"])) for _, r in failures.iterrows())
        df = df[df.apply(lambda r: (r["scenario"], int(r["seed"])) in fail_keys, axis=1)]

    heat = np.zeros((128, 128), dtype=float)

    for _, row in df.iterrows():
        x = int(np.clip(round(row["x"]), 0, 127))
        y = int(np.clip(round(row["y"]), 0, 127))
        heat[y, x] += 1

    world = get_world_for_scenario(scenario_name or "training_1")

    plt.figure(figsize=(8, 8))
    plt.imshow(world, origin="lower", alpha=0.25)
    plt.imshow(np.log1p(heat), origin="lower", alpha=0.75)
    plt.colorbar(label="log(1 + visits)")
    title = "Visitation heatmap"
    if scenario_name:
        title += f" | {scenario_name}"
    if only_failures:
        title += " | failures only"
    plt.title(title)
    plt.xlim(0, 127)
    plt.ylim(0, 127)

    fname = "heatmap"
    if scenario_name:
        fname += f"_{scenario_name}"
    if only_failures:
        fname += "_failures"
    plt.savefig(RESULTS_DIR / f"{fname}.png", dpi=150, bbox_inches="tight")
    plt.show()

for scenario_name in SCENARIOS:
    visitation_heatmap(all_traj, scenario_name=scenario_name, only_failures=False)

if len(failures) > 0:
    for scenario_name in SCENARIOS:
        visitation_heatmap(all_traj, scenario_name=scenario_name, only_failures=True)



# 14. Analyse des actions

On cherche à savoir :

- l'agent utilise-t-il trop `Stay` ?
- utilise-t-il trop les actions Sud ?
- change-t-il trop souvent d'action ?


In [ ]:

if len(all_traj) > 0:
    action_counts = (
        all_traj.groupby(["scenario", "action"])
        .size()
        .reset_index(name="count")
    )

    action_counts["action_name"] = action_counts["action"].map(ACTION_NAMES)

    display(action_counts)

    for scenario_name in SCENARIOS:
        sub = action_counts[action_counts["scenario"] == scenario_name]
        if len(sub) == 0:
            continue

        plt.figure(figsize=(8, 4))
        plt.bar(sub["action_name"], sub["count"])
        plt.title(f"Action distribution | {scenario_name}")
        plt.xlabel("Action")
        plt.ylabel("Count")
        plt.grid(axis="y")
        plt.savefig(RESULTS_DIR / f"action_distribution_{scenario_name}.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No trajectory data.")



# 15. Analyse des épisodes lents

On compare les épisodes rapides et lents pour repérer :

- trajectoires trop longues ;
- vitesse nord faible ;
- mauvais choix de côté ;
- oscillations.


In [ ]:

def add_episode_metrics(traj_df):
    if len(traj_df) == 0:
        return pd.DataFrame()

    rows = []
    for (scenario_name, seed), sub in traj_df.groupby(["scenario", "seed"]):
        sub = sub.sort_values("step")
        actions = sub["action"].to_numpy()

        action_changes = int(np.sum(actions[1:] != actions[:-1])) if len(actions) > 1 else 0

        rows.append({
            "scenario": scenario_name,
            "seed": int(seed),
            "mean_vy": sub["vy"].mean(),
            "mean_abs_vx": sub["vx"].abs().mean(),
            "mean_wy": sub["wy"].mean(),
            "mean_abs_wx": sub["wx"].abs().mean(),
            "action_changes": action_changes,
            "stay_rate": np.mean(actions == 8),
            "south_rate": np.mean(np.isin(actions, [3, 4, 5])),
            "east_rate": np.mean(np.isin(actions, [1, 2, 3])),
            "west_rate": np.mean(np.isin(actions, [5, 6, 7])),
        })

    return pd.DataFrame(rows)

episode_metrics = add_episode_metrics(all_traj)
episodes_with_metrics = episodes.merge(
    episode_metrics,
    on=["scenario", "seed"],
    how="left",
)

display(episodes_with_metrics.sort_values("steps", ascending=False).head(15))

metrics_path = RESULTS_DIR / "episode_metrics.csv"
episodes_with_metrics.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)



# 16. Résumé automatique des problèmes probables

Cette cellule donne des pistes concrètes pour retourner améliorer `04_parameter_tuning.ipynb`.


In [ ]:

def diagnostic_report(df):
    lines = []

    if len(df) == 0:
        return ["Aucun épisode disponible."]

    success_rate = df["success"].mean()
    collision_rate = df["collided"].mean()
    mean_steps_success = df.loc[df["success"], "steps"].mean() if df["success"].any() else np.nan

    lines.append(f"Success rate global: {success_rate:.2%}")
    lines.append(f"Collision rate global: {collision_rate:.2%}")
    lines.append(f"Mean steps among successes: {mean_steps_success:.2f}")

    if collision_rate > 0:
        lines.append("- Des collisions existent : augmenter `collision_penalty` et/ou `center_penalty`.")
        lines.append("- Vérifier si les waypoints passent trop près de l'île.")

    if success_rate < 1.0:
        lines.append("- Certains épisodes échouent : regarder les heatmaps failures.")
        lines.append("- Tester un horizon plus grand ou des waypoints plus conservateurs.")

    if np.isfinite(mean_steps_success) and mean_steps_success > 65:
        lines.append("- Les succès sont lents : renforcer `north_speed_weight` ou réduire des détours excessifs.")
        lines.append("- Comparer gauche/droite selon le vent : le choix de côté peut être trop simpliste.")

    if "stay_rate" in df.columns and df["stay_rate"].mean() > 0.10:
        lines.append("- L'agent utilise trop Stay : pénaliser l'immobilité dans la fonction de score.")

    if "south_rate" in df.columns and df["south_rate"].mean() > 0.10:
        lines.append("- L'agent utilise beaucoup d'actions vers le sud : renforcer progression vers le goal.")

    return lines

report_lines = diagnostic_report(episodes_with_metrics)

print("\n".join(report_lines))

report_path = RESULTS_DIR / "diagnostic_report.txt"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("\nSaved:", report_path)



# 17. Que faire après ce notebook ?

Si tu observes :

## Collisions
Retourne dans `04_parameter_tuning.ipynb` et augmente :
- `collision_penalty`,
- `center_penalty`,
- la distance latérale des waypoints.

## Épisodes lents mais réussis
Ajuste :
- `north_speed_weight`,
- `goal_weight`,
- les waypoints hauts.

## Échecs sans collision
Ajuste :
- `horizon`,
- stratégie gauche/droite,
- pénalités de bord,
- scoring du vent.

## Très bon résultat sur publics
Passe au notebook :

```text
07_build_submission.ipynb
```

pour générer le ZIP final.
